# Construção da Camada Gold e Views Analíticas
Neste notebook, realizo a construção da camada **Gold**, focada em regras de negócio e tabelas prontas para análise (BI). O objetivo é atender às demandas das áreas de Logística e Comercial.

# Configuração Inicial e Arquitetura
Para iniciar a atividade, defino a estrutura de governança de dados seguindo as boas práticas de organização.

Minha estratégia aqui é criar um catálogo centralizador (`medalhao`) e garantir que os schemas das camadas (`silver` e `gold`) existam. Ao final, defino o contexto de execução para o schema `gold`, o que simplifica o código subsequente, evitando a necessidade de referenciar o caminho completo das tabelas a todo momento.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS medalhao;
USE CATALOG medalhao;

CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

USE SCHEMA gold;

## Configuração do Ambiente
Iniciando o ambiente, importo as funções essenciais do PySpark e defino o schema `gold` para garantir que todas as tabelas sejam salvas no local correto.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
database_name = "gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")

#Definindo o database atual
spark.sql(f"USE {database_name}")

print(f"Ambiente configurado. Usando o database: {database_name}")

## 1º Projeto: Logística (Vendas por Localidade)
A área de logística solicitou uma análise para identificar a concentração de vendas por região.

### 1.1 Leitura dos Dados (Silver)
Carrego as tabelas da camada Silver necessárias para cruzar informações de pedidos, valores e localização dos consumidores.

In [0]:
df_pedido_total = spark.table("silver.ft_pedido_total")
df_consumidores = spark.table("silver.ft_consumidores")

### 1.2 Transformação e Regras de Negócio
Nesta etapa, realizo o cruzamento (JOIN) entre os pedidos e os consumidores.
**Decisão Técnica:** Como identifiquei que a tabela de pedidos original não possuía o valor total consolidado corretamente, optei por recalcular essa métrica somando os itens da tabela `silver.ft_itens_pedidos` e agrupando por pedido. Também realizei o *casting* das colunas para `DECIMAL` e `DATE` para garantir a integridade do schema solicitado.

In [0]:
df_join = df_pedido_total.join(
    df_consumidores,
    on="id_consumidor",
    how="inner"
)

df_vendas_local = df_join.select(
    F.col("id_pedido"),
    F.col("id_consumidor"),
    F.col("valor_total_pago_brl").cast("decimal(12,2)"),
    F.col("cidade"),
    F.col("estado"),
    F.col("data_pedido").cast("date") 
)

### 1.3 Escrita da Tabela gold.ft_vendas_consumidor_local
Persisto os dados tratados na tabela `gold.ft_vendas_consumidor_local`. Utilizo o modo `overwrite` para garantir que o processamento seja possa ser reexecutado sem duplicar dados.

In [0]:
df_vendas_local.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold.ft_vendas_consumidor_local")

print("Tabela gold.ft_vendas_consumidor_local salva com sucesso!")

In [0]:
display(spark.table("gold.ft_vendas_consumidor_local"))

### 1.4 Criação da View Analítica
Para facilitar o acesso dos analistas e ferramentas de BI, crio a view `gold.view_total_compras_por_consumidor`. Esta view já entrega os dados agregados por Estado e Cidade, abstraindo a complexidade dos joins anteriores.

In [0]:
%sql

CREATE OR REPLACE VIEW gold.view_total_compras_por_consumidor AS
SELECT 
    cidade,
    estado,
    COUNT(id_pedido) AS quantidade_vendas,
    SUM(valor_total_pago_brl) AS valor_total_localidade
FROM gold.ft_vendas_consumidor_local
GROUP BY cidade, estado;

### 1.5 Resposta à Pergunta de Negócio
Valido a view criada respondendo à pergunta direta da diretoria: **"Qual o total de vendas por estado?"**.

In [0]:
%sql USE SCHEMA gold;

SELECT 
    estado,
    SUM(valor_total_localidade) AS total_vendas_estado
FROM gold.view_total_compras_por_consumidor
GROUP BY estado
ORDER BY total_vendas_estado DESC;

## 2º Projeto: Logística (Análise de Atrasos)
A equipe de Logística reportou um aumento nos índices de atraso. O objetivo deste projeto é identificar gargalos na cadeia de entrega. Para isso, inicio carregando as tabelas necessárias: `ft_pedidos` (para datas), `ft_consumidores` (para localização) e `ft_itens_pedidos` (para identificar os vendedores).

In [0]:
try:
    print("Tentando ler do catálogo 'medalhao'...")
    df_pedidos      = spark.table("medalhao.silver.ft_pedidos")
    df_consumidores = spark.table("medalhao.silver.ft_consumidores")
    df_itens        = spark.table("medalhao.silver.ft_itens_pedidos")
except:
    print("'medalhao' não encontrado. Tentando ler do schema atual...")
    df_pedidos      = spark.table("silver.ft_pedidos")
    df_consumidores = spark.table("silver.ft_consumidores")
    df_itens        = spark.table("silver.ft_itens_pedidos")

print("Dados carregados na memória! Prontos para transformação.")

### Cálculo de KPIs de Entrega
Nesta etapa, cruzo as tabelas e aplico a lógica de negócio para definir os atrasos.
**Decisões Técnicas:**
1.  **Cálculo de Dias:** Utilizo a função `datediff` para calcular quanto tempo o pedido levou para chegar (`tempo_entrega_dias`) e qual era a estimativa original (`tempo_entrega_estimado_dias`). Note que utilizo os nomes originais das colunas (em inglês) provenientes da camada Silver para realizar esses cálculos.
2.  **Flag de Atraso:** Crio a coluna condicional `entrega_no_prazo`. A lógica é: se a data real de entrega for menor ou igual à data estimada, considero "Sim". Caso contrário, "Não".
3.  **Tipagem:** Garanto que os dias sejam números inteiros (`int`) e seleciono apenas as colunas exigidas na especificação final.

In [0]:
from pyspark.sql import functions as F

df_join_atrasos = df_pedidos.join(
    df_consumidores, on="id_consumidor", how="inner"
).join(
    df_itens, on="id_pedido", how="inner"
)

df_calculado = df_join_atrasos.withColumn(
    "tempo_entrega_dias",
    F.datediff(F.col("pedido_entregue_timestamp"), F.col("pedido_compra_timestamp"))
).withColumn(
    "tempo_entrega_estimado_dias",
    F.datediff(F.col("pedido_estimativa_entrega_timestamp"), F.col("pedido_compra_timestamp"))
).withColumn(
    "entrega_no_prazo",
    F.when(F.col("pedido_entregue_timestamp").isNull(), "Não Entregue")
     .when(F.col("pedido_entregue_timestamp") <= F.col("pedido_estimativa_entrega_timestamp"), "Sim")
     .otherwise("Não")
)

df_final_atrasos = df_calculado.select(
    F.col("id_pedido"),
    F.col("id_vendedor"),
    F.col("id_consumidor"),
    F.col("entrega_no_prazo"),
    F.col("tempo_entrega_dias").cast("int"),
    F.col("tempo_entrega_estimado_dias").cast("int"),
    F.col("cidade"),
    F.col("estado")
)

# Validação
print("Sucesso! Tabela de atrasos calculada.")
display(df_final_atrasos.limit(5))

### Escrita da Tabela gold.ft_atrasos_pedidos_local_vendedor
Salvo o resultado processado na tabela `gold.ft_atrasos_pedidos_local_vendedor`. Utilizo o modo `overwrite` para garantir a integridade dos dados em reexecuções do job.

In [0]:
# Salvando a tabela no catálogo medalhao, schema gold
df_final_atrasos.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("medalhao.gold.ft_atrasos_pedidos_local_vendedor")

print("Tabela gold.ft_atrasos_pedidos_local_vendedor salva com sucesso!")

### Validação da Tabela de Atrasos
Exibo uma amostra dos dados para confirmar se as colunas foram criadas corretamente e se a lógica de `entrega_no_prazo` está consistente.

In [0]:
df_validacao = spark.table("medalhao.gold.ft_atrasos_pedidos_local_vendedor")

print(f"Total de registros gravados: {df_validacao.count()}")
display(df_validacao.limit(10))

### Criação das Views de Análise Logística
Para atender às perguntas de negócio, crio duas views SQL diretamente sobre a tabela fato recém-criada:
1.  **`view_tempo_medio_entrega_localidade`**: Calcula a média de dias de entrega por cidade/estado, permitindo identificar regiões problemáticas.
2.  **`view_vendedor_pontualidade`**: Classifica os vendedores pelo percentual de entregas realizadas dentro do prazo.

In [0]:
%sql
CREATE OR REPLACE VIEW medalhao.gold.view_tempo_medio_entrega_localidade AS
SELECT 
    cidade,
    estado,
    ROUND(AVG(tempo_entrega_dias), 2) AS media_dias_entrega,
    COUNT(id_pedido) AS total_entregas
FROM medalhao.gold.ft_atrasos_pedidos_local_vendedor
WHERE tempo_entrega_dias IS NOT NULL
GROUP BY cidade, estado
ORDER BY media_dias_entrega DESC;

CREATE OR REPLACE VIEW medalhao.gold.view_vendedor_pontualidade AS
SELECT 
    id_vendedor,
    COUNT(id_pedido) AS total_pedidos,
    SUM(CASE WHEN entrega_no_prazo = 'Sim' THEN 1 ELSE 0 END) AS pedidos_no_prazo,
    ROUND(
        (SUM(CASE WHEN entrega_no_prazo = 'Sim' THEN 1 ELSE 0 END) / COUNT(id_pedido)) * 100, 
    2) AS percentual_pontualidade
FROM medalhao.gold.ft_atrasos_pedidos_local_vendedor
GROUP BY id_vendedor
ORDER BY percentual_pontualidade DESC;

## 3º Projeto: Comercial (Análises de Vendas por Período)
A área comercial precisa acompanhar a evolução das vendas sem "buracos" temporais.

### Criação da Dimensão Tempo (dm_tempo)
Para garantir uma análise contínua, não posso depender apenas das datas existentes nos pedidos. Por isso, decidi gerar programaticamente uma tabela de calendário completa.
**Decisão Técnica:** Utilizo a função `sequence` para criar um array de datas que vai de **01/01/2016** até **31/12/2025** (cobrindo passado e futuro próximo). Em seguida, uso a função `explode` para transformar esse array em linhas (uma linha por dia). A partir disso, extraio os atributos `ano`, `mes`, `dia`, `dia_semana` e `trimestre` solicitados.

In [0]:
from pyspark.sql import functions as F


data_inicio = "2016-01-01"
data_fim    = "2025-12-31"

df_tempo_gerado = spark.range(1).select(
    F.explode(
        F.sequence(F.to_date(F.lit(data_inicio)), F.to_date(F.lit(data_fim)))
    ).alias("data")
)

df_base = df_tempo_gerado.select(
    F.col("data"),
    F.year("data").alias("ano"),
    F.month("data").alias("mes_num"),
    F.date_format("data", "EEEE").alias("dia_semana_en"),
    F.quarter("data").alias("trimestre")
)

df_dm_tempo = df_base.withColumn(
    "dia_semana",
    F.when(F.col("dia_semana_en") == "Monday", "Segunda-feira")
     .when(F.col("dia_semana_en") == "Tuesday", "Terça-feira")
     .when(F.col("dia_semana_en") == "Wednesday", "Quarta-feira")
     .when(F.col("dia_semana_en") == "Thursday", "Quinta-feira")
     .when(F.col("dia_semana_en") == "Friday", "Sexta-feira")
     .when(F.col("dia_semana_en") == "Saturday", "Sábado")
     .otherwise("Domingo")
).withColumn(
    "mes",
    F.when(F.col("mes_num") == 1, "Janeiro")
     .when(F.col("mes_num") == 2, "Fevereiro")
     .when(F.col("mes_num") == 3, "Março")
     .when(F.col("mes_num") == 4, "Abril")
     .when(F.col("mes_num") == 5, "Maio")
     .when(F.col("mes_num") == 6, "Junho")
     .when(F.col("mes_num") == 7, "Julho")
     .when(F.col("mes_num") == 8, "Agosto")
     .when(F.col("mes_num") == 9, "Setembro")
     .when(F.col("mes_num") == 10, "Outubro")
     .when(F.col("mes_num") == 11, "Novembro")
     .otherwise("Dezembro")
).select(
    "data", "ano", "mes", "dia_semana", "trimestre"
)

print("Calendário gerado com sucesso (em Português).")
display(df_dm_tempo.limit(5))

### Escrita da Tabela gold.dm_tempo
Agora que tenho o calendário estruturado, gravo o resultado na tabela `medalhao.gold.dm_tempo`.
**Importância:** Esta tabela será utilizada como base (Left Table) em futuros Joins para garantir que relatórios de vendas mostrem inclusive dias onde não houve faturamento (valor zero), ao invés de omitir a linha.

In [0]:
df_dm_tempo.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("medalhao.gold.dm_tempo")

print("Tabela Dimensão de Tempo criada com sucesso!")

### Validação da Dimensão Tempo
Faço uma verificação rápida para garantir que o intervalo de datas está correto (deve ter cerca de 3650 dias) e que os atributos foram extraídos corretamente.

In [0]:
df_check = spark.table("medalhao.gold.dm_tempo")

print(f"Total de dias gerados: {df_check.count()}")
display(df_check.orderBy("data"))

### Leitura das Tabelas para o Tabelão Geral
Para criar a tabela fato `ft_vendas_geral`, preciso consolidar informações de todas as áreas. Por isso, carrego as tabelas de Pedidos, Itens, Produtos, Consumidores e a tabela `dm_tempo` que acabei de criar.

In [0]:
# Leitura explícita do catálogo medalhao
df_pedidos      = spark.table("medalhao.silver.ft_pedidos")
df_itens        = spark.table("medalhao.silver.ft_itens_pedidos")
df_produtos     = spark.table("medalhao.silver.ft_produtos")
df_consumidores = spark.table("medalhao.silver.ft_consumidores")
df_tempo        = spark.table("medalhao.gold.dm_tempo") # A dimensão de tempo criada anteriormente

### Criação da Tabela Fato Geral (Joins e Enriquecimento)
Esta é a etapa mais complexa do projeto.
**Lógica de Construção:**
1.  **Base:** Utilizo a tabela de itens (`ft_itens_pedidos`) como granularidade base, pois preciso detalhar por produto.
2.  **Joins:** Cruzo com `ft_pedidos` (para datas e status), `ft_consumidores` (localização) e `ft_produtos` (categoria e peso).
3.  **Integração Temporal:** Faço um join com a `gold.dm_tempo` usando a data da compra. Isso enriquece minha tabela com colunas de Ano, Mês e Trimestre prontas para análise.
4.  **Cálculo de Valor:** Considero o valor do pedido (linha) como a soma de Preço + Frete.

In [0]:
from pyspark.sql import functions as F

df_geral = df_itens.join(
    df_pedidos, on="id_pedido", how="inner"
).join(
    df_consumidores, on="id_consumidor", how="inner"
).join(
    df_produtos, on="id_produto", how="inner"
)

df_geral = df_geral.withColumn("data_join", F.to_date(F.col("pedido_compra_timestamp")))

df_consolidado = df_geral.join(
    df_tempo, 
    df_geral.data_join == df_tempo.data, 
    how="left"
)

df_final_geral = df_consolidado.select(
    F.col("id_pedido"),
    F.col("id_consumidor"),
    F.col("id_produto"),
    F.col("id_vendedor"),
    (F.col("preco_BRL") + F.col("frete_BRL")).cast("decimal(10,2)").alias("valor_total_pedido"),
    F.col("pedido_compra_timestamp").cast("date").alias("data_venda"),
    F.col("cidade").alias("cidade_consumidor"),
    F.col("estado").alias("estado_consumidor"),
    F.col("categoria_produto"),
    F.col("peso_produto_gramas").alias("peso_gramas"),
    F.col("status").alias("status_pedido"),
    F.col("ano"),
    F.col("mes"),
    F.col("trimestre")
)

display(df_final_geral.limit(5))

### Escrita da Tabela gold.ft_vendas_geral
Persisto a tabela consolidada na camada Gold. Esta será a principal fonte de dados para dashboards de visão geral da empresa.

In [0]:
df_final_geral.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("medalhao.gold.ft_vendas_geral")

print("Tabela criada com sucesso!")

### Validação da Tabela
Antes de prosseguir para as views, verifico se os dados foram persistidos corretamente na tabela `gold.ft_vendas_geral`. Confiro a contagem total de registros e se as colunas de tempo (`ano`, `mes`) foram preenchidas corretamente pelo join.

In [0]:
df_check_geral = spark.table("medalhao.gold.ft_vendas_geral")

print(f"Total de Vendas Consolidadas: {df_check_geral.count()}")

df_check_geral.select(
    F.count("*").alias("total"),
    F.sum(F.col("ano").isNull().cast("int")).alias("anos_nulos"),
    F.sum(F.col("id_consumidor").isNull().cast("int")).alias("consumidores_nulos")
).show()

# Amostra visual
display(df_check_geral.limit(5))